# Capítulo 8 · Máquina de Vectores de Soporte Cuántica (QSVM)

## Objetivos

1. Comprender el mapa de características cuántico como codificación de datos en el espacio de Hilbert.
2. Implementar un kernel cuántico con Qiskit Machine Learning.
3. Entrenar y evaluar un clasificador SVM con kernel cuántico en un dataset sintético.

---

## 8.1 Kernel cuántico

El kernel cuántico entre dos puntos $\mathbf{x}, \mathbf{x}' \in \mathbb{R}^d$ se define como el solapamiento del estado de sus mapas de características:

$$K(\mathbf{x}, \mathbf{x}') = |\langle \phi(\mathbf{x}) | \phi(\mathbf{x}') \rangle|^2 = |\langle 0| U^\dagger(\mathbf{x}) U(\mathbf{x}') |0\rangle|^2$$

donde $U(\mathbf{x})$ es un circuito cuántico paramétrico que depende de los datos.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_aer import AerSimulator
from qiskit.primitives import Sampler

print('Módulos ML cuántico cargados.')

In [ ]:
# ── Dataset: dos lunas ────────────────────────────────────────────
np.random.seed(42)
X, y = make_moons(n_samples=100, noise=0.15)

# Normalizar a [0, π]
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

print(f'Dataset: {len(X_train)} train, {len(X_test)} test')
print(f'Características: {X.shape[1]} (2D → 2 qubits)')

# Visualización del dataset
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X[y==0, 0], X[y==0, 1], c='#58a6ff', label='Clase 0', alpha=0.7)
ax.scatter(X[y==1, 0], X[y==1, 1], c='#f78166', label='Clase 1', alpha=0.7)
ax.set_title('Dataset: dos lunas (clasificación binaria)')
ax.legend()
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
plt.tight_layout()
plt.show()

In [ ]:
# ── Mapa de características ZZFeatureMap ──────────────────────────
n_features = X.shape[1]   # 2
feature_map = ZZFeatureMap(
    feature_dimension=n_features,
    reps=2,
    entanglement='linear',
)

print('Mapa de características cuántico (ZZFeatureMap):')
print(feature_map.decompose().draw('text'))

In [ ]:
# ── Kernel cuántico y SVM ─────────────────────────────────────────
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()
qkernel = FidelityQuantumKernel(feature_map=feature_map)

# Calcular matrices de kernel
print('Calculando matrices de kernel cuántico…')
K_train = qkernel.evaluate(x_vec=X_train)
K_test  = qkernel.evaluate(x_vec=X_test, y_vec=X_train)

# SVM con kernel precomputado
svm_quantum = SVC(kernel='precomputed', C=1.0)
svm_quantum.fit(K_train, y_train)

# Evaluación
y_pred_q = svm_quantum.predict(K_test)
acc_q = accuracy_score(y_test, y_pred_q)

# Comparación con SVM clásico RBF
svm_classic = SVC(kernel='rbf', C=1.0)
svm_classic.fit(X_train, y_train)
y_pred_c = svm_classic.predict(X_test)
acc_c = accuracy_score(y_test, y_pred_c)

print(f'\n=== Resultados ===')
print(f'Exactitud QSVM  (kernel cuántico ZZ): {acc_q:.4f}')
print(f'Exactitud SVM   (kernel RBF clásico): {acc_c:.4f}')

## 8.2 Ejercicios propuestos

1. Prueba el mapa de características `PauliFeatureMap` con distintos operadores de Pauli. ¿Cómo afecta a la exactitud?

2. Aumenta el número de reps del `ZZFeatureMap` a 3 y 4. ¿Hay sobreajuste?

3. Aplica QSVM al dataset Iris (4 características, 3 clases) usando un mapa de 4 qubits.